In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Dataset Load
iris = load_iris()
X = iris.data
y = iris.target

print("Setup Complete & Dataset Loaded Successfully!")

Setup Complete & Dataset Loaded Successfully!


In [4]:
# Q1. KNN Manual Tuning
X_train1, X_test1, y_train1, y_test1 = train_test_split(X, y, test_size=0.33, random_state=42)

print("--- Q1: KNN Results ---")
for k in [3, 5, 7, 11, 13, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train1, y_train1)
    print(f"K = {k} | Accuracy: {knn.score(X_test1, y_test1):.4f}")

--- Q1: KNN Results ---
K = 3 | Accuracy: 0.9800
K = 5 | Accuracy: 0.9800
K = 7 | Accuracy: 0.9800
K = 11 | Accuracy: 1.0000
K = 13 | Accuracy: 1.0000
K = 15 | Accuracy: 1.0000


In [5]:
#  Q2. SVM Manual Tuning
print("\n--- Q2: SVM Manual Results ---")
for c in [1, 10, 20]:
    for kernel in ['linear', 'rbf']:
        svm = SVC(C=c, kernel=kernel)
        svm.fit(X_train1, y_train1)
        print(f"C={c}, Kernel='{kernel}' | Accuracy: {svm.score(X_test1, y_test1):.4f}")



--- Q2: SVM Manual Results ---
C=1, Kernel='linear' | Accuracy: 1.0000
C=1, Kernel='rbf' | Accuracy: 1.0000
C=10, Kernel='linear' | Accuracy: 1.0000
C=10, Kernel='rbf' | Accuracy: 0.9800
C=20, Kernel='linear' | Accuracy: 0.9800
C=20, Kernel='rbf' | Accuracy: 1.0000


In [6]:
# Q3. Grid Search CV (SVM)
print("\n--- Q3: GridSearchCV (SVM) ---")
param_grid = {'C': [1, 10, 20], 'kernel': ['linear', 'rbf']}
grid_svm = GridSearchCV(SVC(), param_grid, cv=5)
grid_svm.fit(X_train1, y_train1)
df_grid = pd.DataFrame(grid_svm.cv_results_)
print(df_grid[['param_C', 'param_kernel', 'mean_test_score']])
print("Best Grid Params:", grid_svm.best_params_)


--- Q3: GridSearchCV (SVM) ---
   param_C param_kernel  mean_test_score
0        1       linear             0.95
1        1          rbf             0.93
2       10       linear             0.93
3       10          rbf             0.94
4       20       linear             0.93
5       20          rbf             0.95
Best Grid Params: {'C': 1, 'kernel': 'linear'}


In [8]:
# Q4. Randomized Search CV (SVM)
print("\n--- Q4: RandomizedSearchCV (SVM) ---")
rand_svm = RandomizedSearchCV(SVC(), param_grid, n_iter=5, cv=5, random_state=42)
rand_svm.fit(X_train1, y_train1)
df_rand = pd.DataFrame(rand_svm.cv_results_)
print(df_rand[['param_C', 'param_kernel', 'mean_test_score']])
print("Best Random Search Params:", rand_svm.best_params_)


--- Q4: RandomizedSearchCV (SVM) ---
   param_C param_kernel  mean_test_score
0        1       linear             0.95
1        1          rbf             0.93
2       20          rbf             0.95
3       10       linear             0.93
4       20       linear             0.93
Best Random Search Params: {'kernel': 'linear', 'C': 1}


In [9]:
# Q5. Bagging Random Forest
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train2, y_train2)
rf_acc = rf.score(X_test2, y_test2)

In [10]:
# Q6. AdaBoost & Gradient Boosting
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train2, y_train2)
ada_acc = ada.score(X_test2, y_test2)

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train2, y_train2)
gb_acc = gb.score(X_test2, y_test2)

In [11]:
# Q7. Boosting XGBoost
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
xgb.fit(X_train2, y_train2)
xgb_acc = xgb.score(X_test2, y_test2)

In [12]:
#  Q8. Hyperparameter Tuning on Random Forest
rf_grid_params = {'n_estimators': [50, 100, 150], 'max_depth': [3, 5, 7]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid_params, cv=5)
rf_grid.fit(X_train2, y_train2)

print(f"Q5 - Random Forest Accuracy: {rf_acc:.4f}")
print(f"Q6 - AdaBoost Accuracy: {ada_acc:.4f}")
print(f"Q6 - Gradient Boosting Accuracy: {gb_acc:.4f}")
print(f"Q7 - XGBoost Accuracy: {xgb_acc:.4f}")
print(f"Q8 - Best RF Params: {rf_grid.best_params_} | Best CV Score: {rf_grid.best_score_:.4f}")

Q5 - Random Forest Accuracy: 0.9000
Q6 - AdaBoost Accuracy: 0.9333
Q6 - Gradient Boosting Accuracy: 0.9667
Q7 - XGBoost Accuracy: 0.9333
Q8 - Best RF Params: {'max_depth': 3, 'n_estimators': 50} | Best CV Score: 0.9583


In [13]:
# Q9. Complete Model Comparison
comparison_df = pd.DataFrame({
    'Model': ['SVM (GridSearch)', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [grid_svm.best_score_, rf_acc, ada_acc, gb_acc, xgb_acc]
})

print("\n--- Model Comparison Table ---")
print(comparison_df.sort_values(by='Accuracy', ascending=False).to_string(index=False))


--- Model Comparison Table ---
            Model  Accuracy
Gradient Boosting  0.966667
 SVM (GridSearch)  0.950000
         AdaBoost  0.933333
          XGBoost  0.933333
    Random Forest  0.900000


In [14]:
# Q10. Save Model
 # (XGBoost)
joblib.dump(xgb, 'best_model.pkl')
print("Model Successfully Saved as 'best_model.pkl'!")

Model Successfully Saved as 'best_model.pkl'!
